# sales_intelligence.ipynb

### Sales KPIs
- Total Sales
- Sales Growth
- Average Basket
- Performance:
B2B vs B2C
Payment Methods
Delivery Methods
Sales Status
- discount impact


This notebook analyses the sales of Energical, and gives all the necessery KPIs and analysis to have clean insights and make business decisions.

In [ ]:
import pandas as pd

import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.subplots import make_subplots

import numpy as np

In [ ]:
clean_transactions=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_transactions.csv")
clean_orders=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_orders.csv")
clean_customers=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_customers.csv")
clean_catalogue=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_catalogue.csv")


In [ ]:
clean_customers.rename(columns={
    "Code Client": "customer_id_stage"
    }, inplace=True)

In [ ]:
valid_sales=clean_orders[clean_orders["order_status"].isin(["Terminée", "Partiellement remboursée"])]

## Sales number

In [ ]:
def sale_number(valid_sales):
    sales_number=valid_sales['order_total'].sum()
    return sales_number

## Sales growth/trend

In [ ]:
def Sales_growth(current_sales_number,previous_sales_number):
    if previous_sales_number==0:
        return None
    else:
        sales_growth=((current_sales_number-previous_sales_number)/previous_sales_number)*100
    return sales_growth

## Average basket value

In [ ]:
def avg_basket_value(clean_customers):
    clean_customers["average_basket"] = (
        clean_customers["total_amount"] / clean_customers["orders_count"]
    )

    return clean_customers

## Sales performance

In [ ]:
#performance per method of payment
def performance_per_mop(Valid_sales):
    performance = (
        Valid_sales
        .groupby("payment_method_group")
        .agg(
            Revenue=("order_total_amount", "sum"),
            Orders=("order_id_stage", "count"),
            Average_Basket=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return performance

In [ ]:
shipping_per_order = (
    clean_transactions[["order_id_stage", "shipping_method"]]
    .drop_duplicates()
)
orders_shipping=(
    valid_sales
    .merge(shipping_per_order, on="order_id_stage", how="left")
)

In [ ]:
#performance per delivery method
def performance_per_delivery_method(orders_shipping):
    performance = (
        orders_shipping.groupby("shipping_method").agg(
            Revenue=("order_total_amount", "sum"),
            Orders=("order_id_stage", "count"),
            Average_Basket=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return performance


In [ ]:
#performance per customer type
def performance_per_customer_type(clean_customers):
    performance=(
        clean_customers.groupby("customer_type_inferred").agg(
            Revenue=("total_amount", "sum"),
            Orders=("orders_count", "count"),
            Average_Basket=("average_basket", "mean")
        )
        .reset_index()
    )
    return performance

## Return rate vs completed sales

In [ ]:
status_mapping = {
    "Terminée": "Completed",
    "En cours": "In Progress",
    "En livraison par NOEST": "In Progress",

    "En attente": "Pending",
    "Attente paiement": "Pending",
    "Partiellement payé": "Pending",

    "Retour NOEST": "Returned",
    "Remboursée": "Refunded",
    "Partiellement remboursée": "Partially Refunded"
}
clean_orders["sales_status"] = clean_orders["order_status"].map(status_mapping)

In [ ]:
def sales_status(clean_orders):

    sales_status = (
        clean_orders.groupby("sales_status")
        .agg(
            orders_count=("order_id_stage", "count")
        )
        .reset_index()
    )

    sales_status["percentage"] = (
        sales_status["orders_count"]
        / sales_status["orders_count"].sum()
    ) * 100

    sales_status["percentage"] = sales_status["percentage"].round(2)

    return sales_status

## discount impact

In [ ]:
clean_transactions.rename(columns={
    "has_negative_price": "has_discount"
}, inplace=True)

In [ ]:
clean_transactions[clean_transactions["has_discount"]==True]

In [ ]:
def discount_impact(clean_transactions, clean_orders):

    # Create one row per order indicating whether it contains a discount
    discount_orders = (
        clean_transactions.groupby("order_id_stage")["has_discount"]
        .any()
        .reset_index()
        .rename(columns={"has_discount": "order_has_discount"})
    )

    # Merge the flag into the orders table
    orders = clean_orders.merge(
        discount_orders,
        on="order_id_stage",
        how="left"
    )

    # Orders with no discount line become False
    orders["order_has_discount"] = orders["order_has_discount"].fillna(False)

    # Aggregate at the order level
    impact = (
        orders.groupby("order_has_discount")
        .agg(
            Total_Revenue=("order_total_amount", "sum"),
            Total_Orders=("order_id_stage", "count"),
            Average_Order_Value=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return impact

In [ ]:
discount_impact(clean_transactions, clean_orders)

## Free delivery impact

In [ ]:
def free_del_impact(clean_transactions, clean_orders):
    free_shipping_orders =(
        clean_transactions.groupby("order_id_stage")["free_shipping"]
                .any()
                .reset_index()
                .rename(columns={"free_shipping": "order_has_free_shipping"})
    )
    
    orders = clean_orders.merge(
        free_shipping_orders,
        on="order_id_stage",
        how="left"
    )
    

    orders["order_has_free_shipping"] = orders["order_has_free_shipping"].fillna(False)
    impact = (
        orders.groupby("order_has_free_shipping")
        .agg(
            Total_Revenue=("order_total_amount", "sum"),
            Total_Orders=("order_id_stage", "count"),
            Average_Order_Value=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return impact

In [ ]:
free_del_impact(clean_transactions, clean_orders)

# Graphs

### sales number

In [ ]:
valid_sales["order_date"]=valid_sales["order_date"].astype("datetime64[ns]")

In [ ]:
sales = (
    valid_sales.groupby(
        valid_sales["order_date"].dt.to_period("M")
    )["order_total_amount"]
    .sum()
    .reset_index()
)
sales["order_date"] = sales["order_date"].astype(str)
fig = px.line(
    sales,
    x="order_date",
    y="order_total_amount",
    markers='markers',
    title="Sales Revenue"
)

fig.show()

## B2B VS B2C graph

In [ ]:
b2b_b2c = (
    clean_customers.groupby("customer_type_inferred")
    .agg(
        Revenue=("total_amount", "sum"),
        Customers=("customer_id_stage", "count"),
        Average_Basket=("average_basket", "mean")
    )
    .reset_index()
)

In [ ]:
perf = performance_per_customer_type(clean_customers)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=perf["customer_type_inferred"], y=perf["Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=perf["customer_type_inferred"], y=perf["Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=perf["customer_type_inferred"], y=perf["Average_Basket"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="B2B vs B2C Performance", showlegend=False)
fig.show()

In [ ]:
perf = performance_per_mop(valid_sales)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=perf["payment_method_group"], y=perf["Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=perf["payment_method_group"], y=perf["Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=perf["payment_method_group"], y=perf["Average_Basket"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="B2B vs B2C Performance", showlegend=False)
fig.show()

In [ ]:
perf = performance_per_delivery_method(orders_shipping)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=perf["shipping_method"], y=perf["Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=perf["shipping_method"], y=perf["Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=perf["shipping_method"], y=perf["Average_Basket"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="B2B vs B2C Performance", showlegend=False)
fig.show()

In [ ]:
sales_status(clean_orders)

In [ ]:
fig= px.pie(
    sales_status(clean_orders),
    names='sales_status',
    values='orders_count',
    
)
fig.show()

In [ ]:
impact = free_del_impact(clean_transactions, clean_orders)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=impact["order_has_free_shipping"], y=impact["Total_Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=impact["order_has_free_shipping"], y=impact["Total_Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=impact["order_has_free_shipping"], y=impact["Average_Order_Value"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="Free Shipping Impact", showlegend=False)
fig.show()

In [ ]:
impact = discount_impact(clean_transactions, clean_orders)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=impact["order_has_discount"], y=impact["Total_Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=impact["order_has_discount"], y=impact["Total_Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=impact["order_has_discount"], y=impact["Average_Order_Value"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="Discount Impact", showlegend=False)
fig.show()